# 第2回 宿題（発展）
用語集から作ったQ&Aで LoRA を学習し、学習前後と、用語集の検索で比べる。

- A: `make_sft_dataset.py` のデータだけで学習
- B: A に「用語集にない語は分からないと答える」例と、別名の質問を足して学習
- 検索: 質問の中に用語か別名があれば定義を返す（学習なし）


In [ ]:
%pip uninstall -y -q torchao
%pip install -q "transformers==5.18.0" "peft==0.21.1" "trl==1.14.2" "accelerate==1.15.0" "datasets==5.1.0"
!nvidia-smi --query-gpu=name --format=csv,noheader
!rm -rf repo && git clone -q -b lec02 https://github.com/Noah104kkk/lecture-ai-engineering-2026.git repo
!cd repo && python scripts/make_sft_dataset.py > ../glossary_sft.jsonl
!wc -l glossary_sft.jsonl

In [ ]:
import json, re, time, random
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
import pandas as pd
pd.set_option("display.max_colwidth", None)

BASE = "Qwen/Qwen3.5-0.8B"
REV = "2fc06364715b967f1860aea9cf38778875588b17"
EPOCHS = 5
MAX_NEW = 160
tok = AutoTokenizer.from_pretrained(BASE, revision=REV)
if tok.pad_token is None: tok.pad_token = tok.eos_token
terms = json.load(open("repo/data/glossary.json", encoding="utf-8"))["terms"]
print(len(terms))

In [ ]:
# 評価用の質問と正解（自分で用意）。must は答えに全部入っていれば正解とする語
SAME = [  # 用語集にある語（学習と同じ言い回し）
    ("「LoRA」とは何ですか？", ["小さな行列"]),
    ("「エポック」とは何ですか？", ["1周"]),
    ("「ガードレール」とは何ですか？", ["制約"]),
    ("「fork」とは何ですか？", ["コピー"]),
    ("「レート制限」とは何ですか？", ["上限"]),
    ("「MCPサーバー」とは何ですか？", ["公開"]),
    ("「過学習」とは何ですか？", ["学習データ"]),
    ("「APIキー」とは何ですか？", ["秘密"]),
]
PARA = [  # 言い回しを変えた質問
    ("LoRAってどんな仕組みで学習するの？", ["小さな行列"]),
    ("学習データを1周することを何と呼びますか？", ["エポック"]),
    ("エージェントが変な行動をしないようにする制約を何と言いますか？", ["ガードレール"]),
    ("他人のrepoを自分のアカウントにコピーすることを何と言いますか？", ["fork"]),
    ("APIを呼びすぎて429エラーが出たのはなぜですか？", ["上限"]),
    ("Model Context Protocolとは何のためのものですか？", ["標準"]),
    ("overfittingが起きるとどうなりますか？", ["未知"]),
    ("APIキーをコードに書いてはいけないのはなぜですか？", ["漏れ"]),
    ("モデルが事実と違うことをもっともらしく言う現象は何ですか？", ["ハルシネーション"]),
    ("Pythonでエージェントを作るフレームワークで第3回の本線は何ですか？", ["Pydantic AI"]),
]
UNKNOWN = [  # 用語集にない語。分からないと答えるのが正解
    "「RAG」とは何ですか？",
    "「リランキング」とは何ですか？",
    "「知識蒸留」とは何ですか？",
    "「ビームサーチ」とは何ですか？",
    "「lec99-end タグ」とは何ですか？",
    "「宿題の5レベル」とは何ですか？",
    "「AIE2026の中間試験」とは何ですか？",
    "「ベクトルデータベース」について簡単に説明してください。",
]
REFUSE = "用語集に記載がありません。確認が必要です。"
REFUSE_WORDS = ["記載がありません", "確認が必要", "わかりません", "分かりません", "存じません"]

EVAL = [("same", q, m) for q, m in SAME] + [("para", q, m) for q, m in PARA] + [("unknown", q, None) for q in UNKNOWN]
# 評価の質問と学習データが重ならない未知語か確認
names = " ".join(t["term"] + " " + " ".join(t["aliases"]) for t in terms)
for q in UNKNOWN:
    w = re.search("「(.+)」", q).group(1)
    assert w not in names, w


def score(kind, m, text):
    if kind == "unknown":
        return any(w in text for w in REFUSE_WORDS)
    return all(w in text for w in m)


def summarize(rows):
    out = {}
    for k in ["same", "para", "unknown"]:
        r = [x for x in rows if x["kind"] == k]
        out[k] = sum(x["ok"] for x in r) / len(r)
    return out


def generate(m, q):
    prompt = tok.apply_chat_template([{"role": "user", "content": q}], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    x = tok(prompt, return_tensors="pt").to(m.device)
    with torch.no_grad():
        y = m.generate(**x, max_new_tokens=MAX_NEW, do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.decode(y[0][x["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def evaluate(m, label):
    t0 = time.time()
    rows = []
    for kind, q, must in EVAL:
        a = generate(m, q)
        rows.append({"label": label, "kind": kind, "q": q, "a": a, "ok": score(kind, must, a)})
    s = summarize(rows)
    print(label, s, round(time.time() - t0), "秒")
    return s, rows


def search(q):
    # 質問の中に用語か別名があれば定義を返す。長い名前を優先
    best = None
    for t in terms:
        for name in [t["term"]] + t["aliases"]:
            if len(name) >= 2 and name.lower() in q.lower():
                if best is None or len(name) > len(best[0]):
                    best = (name, t["definition"])
    return best[1] if best else REFUSE


search_rows = [{"label": "search", "kind": k, "q": q, "a": search(q), "ok": score(k, m, search(q))} for k, q, m in EVAL]
search_sum = summarize(search_rows)
print("search", search_sum)

In [ ]:
def load():
    m = AutoModelForCausalLM.from_pretrained(BASE, revision=REV, dtype=torch.float32).to("cuda")
    m.config.use_cache = True
    return m


def train(records, name):
    m = load()
    m.config.use_cache = False
    lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                      target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
    args = SFTConfig(output_dir=f"ckpt_{name}", num_train_epochs=EPOCHS, per_device_train_batch_size=4, gradient_accumulation_steps=2,
                     learning_rate=2e-4, lr_scheduler_type="cosine", warmup_steps=6, logging_steps=20, save_strategy="no",
                     max_length=512, packing=False, report_to="none", seed=42)
    tr = SFTTrainer(model=m, args=args, train_dataset=Dataset.from_list(records), peft_config=lora, processing_class=tok)
    t0 = time.time()
    tr.train()
    sec = round(time.time() - t0)
    loss = [x["loss"] for x in tr.state.log_history if "loss" in x][-1]
    print(name, "学習", sec, "秒 / steps", tr.state.global_step, "/ 最終loss", loss)
    t = tr.model
    t.config.use_cache = True
    t.eval()
    return t, {"seconds": sec, "steps": tr.state.global_step, "final_loss": loss, "n_train": len(records)}


base = load()
before_sum, before_rows = evaluate(base, "before")
del base
torch.cuda.empty_cache()

In [ ]:
# A: スクリプトのデータだけ
data_a = [json.loads(l) for l in open("glossary_sft.jsonl", encoding="utf-8")]
tuned_a, info_a = train(data_a, "A")
a_sum, a_rows = evaluate(tuned_a, "A")
del tuned_a
torch.cuda.empty_cache()

In [ ]:
# B: A に別名の質問と、用語集にない語を断る例を足す（評価の未知語は使わない）
def pair(q, a):
    return {"messages": [{"role": "user", "content": q}, {"role": "assistant", "content": a}]}

data_b = list(data_a)
for t in terms:
    for al in t["aliases"][:1]:
        data_b.append(pair(f"「{al}」とは何ですか？", t["definition"]))
OTHER = ["勾配降下法", "活性化関数", "畳み込みニューラルネットワーク", "バッチ正規化", "敵対的生成ネットワーク", "決定木", "ランダムフォレスト", "主成分分析",
         "転移学習", "位置エンコーディング", "自己教師あり学習", "lec50-start タグ", "宿題の10レベル", "AIE2026の期末試験", "AIE2026の合宿", "クロスバリデーション",
         "正則化", "ドロップアウト", "残差接続", "マルチヘッド注意"]
assert not [w for w in OTHER if w in names]
for w in OTHER:
    data_b.append(pair(f"「{w}」とは何ですか？", REFUSE))
    data_b.append(pair(f"{w}について簡単に説明してください。", REFUSE))
random.Random(42).shuffle(data_b)
tuned_b, info_b = train(data_b, "B")
b_sum, b_rows = evaluate(tuned_b, "B")
del tuned_b
torch.cuda.empty_cache()

In [ ]:
table = pd.DataFrame([dict(label="学習前", **before_sum), dict(label="A 用語集のみ", **a_sum), dict(label="B 断る例も追加", **b_sum), dict(label="検索", **search_sum)]).set_index("label")
display(table)
detail = pd.DataFrame([{"kind": b["kind"], "質問": b["q"], "学習前": b["a"][:70], "前": b["ok"], "A": a["a"][:70], "A正": a["ok"], "B": x["a"][:70], "B正": x["ok"], "検索": s["ok"]}
                       for b, a, x, s in zip(before_rows, a_rows, b_rows, search_rows)])
display(detail)
result = {"base": BASE, "revision": REV, "epochs": EPOCHS, "max_new_tokens": MAX_NEW, "train": {"A": info_a, "B": info_b},
          "summary": {"before": before_sum, "A": a_sum, "B": b_sum, "search": search_sum},
          "rows": before_rows + a_rows + b_rows + search_rows, "executed_training_myself": True}
json.dump(result, open("advanced_result.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print("RESULT_JSON_START")
print(json.dumps({k: v for k, v in result.items() if k != "rows"}, ensure_ascii=False))
print("RESULT_JSON_END")